# 07 Ensemble Bagged Model
Bagged Ensemble (6x Bootstrap Aggregation of CatBoost)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error, explained_variance_score

# Set directories
BASE_DIR = r"c:\Users\BG\Desktop\Codes"
DATA_PATH = os.path.join(BASE_DIR, "Files to Analyze", "Ensemble_dataset.csv")
STEP4_DIR = os.path.join(BASE_DIR, "Ensemble_PCC_Project_Results", "Step4_Data_Preprocessing")

# 1. Load Data
df = pd.read_csv(DATA_PATH)
TARGET_COLS = ["co2_recovery", "co2_purity", "sec_energy", "solvent_loss", "opex_usd_per_t", "corrosion_rate"]
INPUT_COLS = [c for c in df.columns if c not in TARGET_COLS]
N_FEATURES = len(INPUT_COLS)
N_TARGETS = len(TARGET_COLS)

print(f"Loaded dataset: {df.shape[0]} samples, {N_FEATURES} inputs, {N_TARGETS} targets.")

# Load preprocessed 80/20 train/test splits & scalers
train_df = pd.read_csv(os.path.join(STEP4_DIR, "train_dataset_80pct.csv"))
test_df = pd.read_csv(os.path.join(STEP4_DIR, "test_dataset_20pct.csv"))
X_tr_scaled = np.load(os.path.join(STEP4_DIR, "X_train_scaled.npy"))
X_te_scaled = np.load(os.path.join(STEP4_DIR, "X_test_scaled.npy"))
y_tr = train_df[TARGET_COLS].values
y_te = test_df[TARGET_COLS].values
scaler_X = joblib.load(os.path.join(STEP4_DIR, "scaler_X.pkl"))

def evaluate_metrics(y_true, y_pred, p=N_FEATURES):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else r2
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-9))) * 100.0
    medae = median_absolute_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    return {"R2": r2, "Adj_R2": adj_r2, "MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE": mape, "MedAE": medae, "EVS": evs}

# 2. Build and Train Bagged Ensemble (6x Bootstrap Replicates of CatBoost-PSO)
from catboost import CatBoostRegressor

print("\n--- Training Bagged Ensemble (6x Bootstrap Replicates) ---")
N_BAGS = 6
bagged_preds = np.zeros_like(y_te)

for i, col in enumerate(TARGET_COLS):
    replicate_preds = np.zeros((len(X_te_scaled), N_BAGS))
    for b in range(N_BAGS):
        # Bootstrap resampling with replacement
        boot_idx = np.random.choice(len(X_tr_scaled), size=len(X_tr_scaled), replace=True)
        m = CatBoostRegressor(
            iterations=350,
            depth=7,
            learning_rate=0.08,
            l2_leaf_reg=3.0,
            random_seed=42 + b,
            verbose=0
        )
        m.fit(X_tr_scaled[boot_idx], y_tr[boot_idx, i])
        replicate_preds[:, b] = m.predict(X_te_scaled)
    # Ensemble mean across bootstrap replicates
    bagged_preds[:, i] = np.mean(replicate_preds, axis=1)
    print(f"Finished Bagging 6x for {col}")

# 3. Evaluate Metrics
results = []
for i, col in enumerate(TARGET_COLS):
    m = evaluate_metrics(y_te[:, i], bagged_preds[:, i])
    m["Target"] = col
    results.append(m)

res_df = pd.DataFrame(results)
print("\n--- Bagged Ensemble (6x) Performance Across 6 Targets ---")
print(res_df[["Target", "R2", "Adj_R2", "RMSE", "MAE", "MAPE"]])
print(f"--> Bagged Ensemble R2: {res_df['R2'].mean():.5f} | MAE: {res_df['MAE'].mean():.2f}")

# 4. Parity Plots
fig, axes = plt.subplots(2, 3, figsize=(15, 9), dpi=200)
for i, ax in enumerate(axes.flatten()):
    ax.scatter(y_te[:, i], bagged_preds[:, i], alpha=0.4, color="#17becf", s=15)
    min_v, max_v = min(y_te[:, i].min(), bagged_preds[:, i].min()), max(y_te[:, i].max(), bagged_preds[:, i].max())
    ax.plot([min_v, max_v], [min_v, max_v], 'r--', lw=1.5, label="1:1 Parity")
    ax.set_title(f"{TARGET_COLS[i]} (R²={res_df.loc[i, 'R2']:.4f})", fontweight="bold")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(True, linestyle=":", alpha=0.6)
plt.suptitle("Ensemble Bagged (6x CatBoost) Parity Plots", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{os.path.splitext(__file__)[0] if '__file__' in locals() else 'parity_plot'}_parity_plots.png", dpi=200, bbox_inches="tight")
plt.close()
print("Parity plot generated and saved successfully.")
